# Patient-level privacy: how well are individual patients protected?

Tests on the released files of one level, `tier2_merge_drop3_pre_epic`, against an attacker described in `PRIVACY_THREAT_MODEL.md`. All results are counts and shares; no rows leave the SDE. A result is a statement about this attacker, not a guarantee.

The attacker holds the released files and nothing else from the SDE (no keys, no site labels, no other level). They want to find a known person's record, learn something about them, or link their stays. Four scenarios describe what they already know:

| | What the attacker knows | What they try to do | Test (below) |
|---|---|---|---|
| **S1** | the target's demographics: age band, sex, weight band, diagnosis group, outcome | find the target's record among the released stays | row `m = 0` of the S2 table |
| **S2** | S1 plus *m* exact measurements from the stay (concept, hour since admission, value), as a clinician or parent might remember | single out the target's record | known-observation attack |
| **S3** | two released stays | decide whether they belong to the same child | repeat-stay linkage |
| **S4** | demographics only | learn the target's outcome (attribute disclosure) | outcome diversity |

The tests are generous to the attacker on one point (exact hours and exact values are known) and limited on others (observations are chosen at random, there is no outside data, and the release holds no dates). The notebook runs the tests in the order S2 (with S1), S3, S4.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))
import patient_report as PR

In [ ]:
derived_dir = open_picu_dir / "derived"
out_root = derived_dir / "anonymised"
reports_dir = derived_dir / "reports"
static = pd.read_parquet(derived_dir / "static" / "pooled_static.parquet")
level = "tier2_merge_drop3_pre_epic"

### S1 and S2: known demographics and known observations

For *m* = 0, 1, 3, 5, 10 known exact observations of 500 randomly chosen stays, how many released stays fit. `m = 0` is S1 (demographics only). `unique` is the share of targets with exactly one candidate; `below_k` is the share with fewer than 10. `att5` repeats the attack with a 5% tolerance on the values.

In [ ]:
att = PR.known_observation_attack(out_root, level)
att5 = PR.known_observation_attack(out_root, level, rel_tol=0.05)
att, att5

### S3: linking repeat stays of one child

For stays that have a sibling stay of the same patient, is a sibling among the nearest 1 or 5 stays in the same split group? `chance` is the hit rate for random guessing and `lift` is `hit` divided by `chance`.

In [ ]:
link = PR.repeat_stay_linkage(out_root, level, static)
link

### S4: outcome diversity

For each stay, the outcomes of the stays that share its quasi-identifiers (its cell). A death in a cell where everyone died has its outcome disclosed to an attacker who knows only the demographics.

In [ ]:
div = PR.outcome_diversity(out_root, level)
div

### S2 on the hourly level

Same attack on `tier2_merge_drop3_pre_epic_hourly` (hours rounded, one row per hour). If uniqueness at m = 3 stays near 99%, coarser time alone does not help and the values are doing the identifying.

In [ ]:
att_h = PR.known_observation_attack(out_root, "tier2_merge_drop3_pre_epic_hourly")
att_h